## Cell 1: Mount Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", DEVICE)

## Cell 2: Import thư viện

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, f1_score

## Cell 3: Cấu hình tham số toàn cục

Định nghĩa đường dẫn, kích thước ảnh, batch size, số epoch.
ResNet50 yêu cầu ảnh tối thiểu 224x224.

In [ ]:
BASE_DIR   = '/content/drive/MyDrive/ML2/Dataset_Split'
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
VALID_DIR  = os.path.join(BASE_DIR, 'val')
TEST_DIR   = os.path.join(BASE_DIR, 'test')

IMG_SIZE    = 224          # ResNet50 yêu cầu 224x224
BATCH_SIZE  = 32
EPOCHS      = 20
NUM_CLASSES = 7
CLASSES     = ['cardboard', 'glass', 'metal', 'organic', 'paper', 'plastic', 'trash']

print("Cấu hình:")
print(f"  IMG_SIZE   : {IMG_SIZE}x{IMG_SIZE}")
print(f"  BATCH_SIZE : {BATCH_SIZE}")
print(f"  EPOCHS     : {EPOCHS}")
print(f"  DEVICE     : {DEVICE}")

## Cell 4: Thống kê dataset

Đếm số ảnh trong mỗi lớp của train, valid, test.

In [ ]:
def count_images(split_dir, split_name):
    print(f"\n{'='*45}")
    print(f"  Tập {split_name.upper()}")
    print(f"{'='*45}")
    print(f"{'Lớp':<15} {'Số ảnh':>10}")
    print(f"{'-'*25}")
    total = 0
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            print(f"{cls:<15} {n:>10}")
            total += n
    print(f"{'-'*25}")
    print(f"{'TỔNG CỘNG':<15} {total:>10}")

count_images(TRAIN_DIR, 'train')
count_images(VALID_DIR, 'valid')
count_images(TEST_DIR,  'test')

## Cell 5: Tạo DataLoader

- Train: augmentation nhẹ + chuẩn hóa theo mean/std của ImageNet
- Valid/Test: chỉ resize và chuẩn hóa, không augment.

In [ ]:
# ImageNet mean và std — bắt buộc khi dùng pretrained model
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Dataset
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = datasets.ImageFolder(VALID_DIR, transform=val_test_transform)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_test_transform)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Class indices:", train_dataset.class_to_idx)
print(f"\nSố batch train : {len(train_loader)}")
print(f"Số batch valid : {len(valid_loader)}")
print(f"Số batch test  : {len(test_loader)}")

## Cell 6: Load ResNet50 Pretrained + Thay đổi Classifier

Transfer Learning:
- Giữ nguyên toàn bộ backbone ResNet50 (đã học đặc trưng từ ImageNet).
- Chỉ thay thế lớp fc (fully connected) cuối cùng thành 7 output.
- Giai đoạn 1: Freeze backbone, chỉ train lớp fc mới.
- Giai đoạn 2 (fine-tune): Unfreeze và train toàn bộ với LR rất nhỏ.

In [ ]:
# Load ResNet50 pretrained
model = models.resnet50(weights='IMAGENET1K_V1')

# Freeze toàn bộ backbone
for param in model.parameters():
    param.requires_grad = False

# Thay lớp fc cuối: 2048  thành 7
model.fc = nn.Sequential(
    nn.Linear(2048, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(DEVICE)

# Chỉ in thông tin lớp fc
print("Lớp FC mới:")
print(model.fc)
print(f"\nTổng params       : {sum(p.numel() for p in model.parameters()):,}")
print(f"Params được train : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Cell 7: Định nghĩa hàm train và evaluate

Tách riêng hàm train_one_epoch và evaluate để dùng lại
cho cả

giai đoạn 1 (frozen) và giai đoạn 2 (fine-tune).

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader, desc="  Training", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total


print("Đã định nghĩa hàm train_one_epoch và evaluate")

## Cell 8: Giai đoạn 1 — Train lớp FC với Backbone Frozen

Chỉ train lớp fc mới thêm vào, backbone ResNet50 giữ nguyên.

Mục tiêu: nhanh chóng học cách phân loại 7 lớp rác
trước khi fine-tune toàn bộ mạng.
Chạy 5 epoch, mỗi epoch rất nhanh vì ít params cần update.

In [ ]:
criterion = nn.CrossEntropyLoss()

# Chỉ optimize lớp fc
optimizer_stage1 = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler_stage1 = optim.lr_scheduler.StepLR(optimizer_stage1, step_size=3, gamma=0.5)

history_s1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
EPOCHS_STAGE1 = 5

print("=" * 60)
print("  GIAI ĐOẠN 1: Train FC Layer (Backbone Frozen)")
print("=" * 60)

best_val_acc = 0.0

for epoch in range(1, EPOCHS_STAGE1 + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_stage1)
    val_loss, val_acc     = evaluate(model, valid_loader, criterion)
    scheduler_stage1.step()

    history_s1['train_loss'].append(train_loss)
    history_s1['train_acc'].append(train_acc)
    history_s1['val_loss'].append(val_loss)
    history_s1['val_acc'].append(val_acc)

    print(f"Epoch [{epoch:2d}/{EPOCHS_STAGE1}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_resnet50.pth')
        print(f" Saved best model (val_acc={val_acc:.4f})")

print(f"\nBest Val Accuracy (Stage 1): {best_val_acc:.4f}")

## Cell 9: Giai đoạn 2 — Fine-tune toàn bộ ResNet50

Unfreeze toàn bộ backbone, train với learning rate rất nhỏ (1e-5)

để tinh chỉnh các đặc trưng đã học mà không làm mất kiến thức cũ.
Đây là bước quyết định accuracy cuối cùng.

In [ ]:
# Unfreeze toàn bộ
for param in model.parameters():
    param.requires_grad = True

print(f"Params được train: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# LR rất nhỏ cho backbone, lớn hơn cho fc
optimizer_stage2 = optim.Adam([
    {'params': [p for name, p in model.named_parameters() if 'fc' not in name],
     'lr': 1e-5},   # backbone: LR nhỏ
    {'params': model.fc.parameters(),
     'lr': 1e-4},   # fc: LR lớn hơn một chút
])

scheduler_stage2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage2, mode='min', factor=0.5, patience=2
)

history_s2 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
EPOCHS_STAGE2 = 20
patience_counter = 0
PATIENCE = 5

print("=" * 60)
print("  GIAI ĐOẠN 2: Fine-tune toàn bộ ResNet50")
print("=" * 60)

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_stage2)
    val_loss, val_acc     = evaluate(model, valid_loader, criterion)
    scheduler_stage2.step(val_loss)

    history_s2['train_loss'].append(train_loss)
    history_s2['train_acc'].append(train_acc)
    history_s2['val_loss'].append(val_loss)
    history_s2['val_acc'].append(val_acc)

    print(f"Epoch [{epoch:2d}/{EPOCHS_STAGE2}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_resnet50.pth')
        print(f"  Saved best model (val_acc={val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n Early stopping tại epoch {epoch}")
            break

print(f"\nBest Val Accuracy (Stage 2): {best_val_acc:.4f}")

cell 10

In [ ]:
# Ghép history 2 giai đoạn
all_train_loss = history_s1['train_loss'] + history_s2['train_loss']
all_val_loss   = history_s1['val_loss']   + history_s2['val_loss']
all_train_acc  = history_s1['train_acc']  + history_s2['train_acc']
all_val_acc    = history_s1['val_acc']    + history_s2['val_acc']

epochs_total = range(1, len(all_train_loss) + 1)
split_epoch  = EPOCHS_STAGE1 + 0.5   # vạch phân chia 2 giai đoạn

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Loss ---
axes[0].plot(epochs_total, all_train_loss, 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs_total, all_val_loss,   'r-o', markersize=4, label='Val Loss')
axes[0].axvline(x=split_epoch, color='gray', linestyle='--', label='Bắt đầu fine-tune')
axes[0].set_title('Loss theo Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Accuracy ---
axes[1].plot(epochs_total, all_train_acc, 'b-o', markersize=4, label='Train Acc')
axes[1].plot(epochs_total, all_val_acc,   'r-o', markersize=4, label='Val Acc')
axes[1].axvline(x=split_epoch, color='gray', linestyle='--', label='Bắt đầu fine-tune')
axes[1].set_title('Accuracy theo Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ResNet50 Transfer Learning — Lịch sử Huấn luyện',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_history_resnet50.png', dpi=150, bbox_inches='tight')
plt.show()

#cell 11

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_resnet50.pth'))
model.eval()

# Inference
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

class_names = list(test_dataset.class_to_idx.keys())
test_acc = (all_preds == all_labels).mean()
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# cell 12

In [ ]:
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_preds,
                             target_names=class_names, digits=4))

macro_f1    = f1_score(all_labels, all_preds, average='macro')
micro_f1    = f1_score(all_labels, all_preds, average='micro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"{'='*60}")
print(f"  Macro    F1 : {macro_f1:.4f}")
print(f"  Micro    F1 : {micro_f1:.4f}")
print(f"  Weighted F1 : {weighted_f1:.4f}")
print(f"{'='*60}")

# cell 13

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix — ResNet50', fontsize=14, fontweight='bold')
plt.ylabel('Nhãn Thực', fontsize=12)
plt.xlabel('Nhãn Dự Đoán', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_resnet50.png', dpi=150, bbox_inches='tight')
plt.show()

# cell 14

In [ ]:
save_path = '/content/drive/MyDrive/ML2/best_resnet50.pth'
torch.save(model.state_dict(), save_path)
print(f"Đã lưu model tại: {save_path}")

# Lưu history
import json
history_all = {
    'stage1': history_s1,
    'stage2': history_s2,
}
with open('/content/drive/MyDrive/ML2/history_resnet50.json', 'w') as f:
    json.dump(history_all, f)
print("Đã lưu history")